# English-Spanish NLP Translator

This notebook uses the **Python 3.14 (NLP Translator)** kernel and keeps Hugging Face datasets on disk.

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CACHE_DIR = PROJECT_ROOT / ".hf_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Python:", sys.version.split()[0])
print("Kernel executable:", sys.executable)
print("Dataset cache:", CACHE_DIR)

Python: 3.14.4
Kernel executable: C:\Users\mdsha\AppData\Local\Python\pythoncore-3.14-64\python.exe
Dataset cache: C:\Users\mdsha\Downloads\Natural Language Processing\.hf_cache


In [2]:
from packaging.version import Version
import datasets
from datasets import load_dataset

if Version(datasets.__version__) < Version("5.0.0"):
    raise RuntimeError(
        f"datasets {datasets.__version__} is too old for Python 3.14. "
        "Upgrade datasets and restart the kernel."
    )

nmt_original_valid_set, nmt_test_set = load_dataset(
    "ageron/tatoeba_mt_train",
    "eng-spa",
    split=["validation", "test"],
    cache_dir=str(CACHE_DIR / "datasets"),
    keep_in_memory=False,
)

split = nmt_original_valid_set.train_test_split(
    train_size=0.8,
    seed=42,
    keep_in_memory=False,
)
nmt_train_set = split["train"]
nmt_valid_set = split["test"]

print("datasets version:", datasets.__version__)
print("train rows:", nmt_train_set.num_rows)
print("validation rows:", nmt_valid_set.num_rows)
print("test rows:", nmt_test_set.num_rows)
print("\nFirst nmt_train_set item:")
print(nmt_train_set[0])

datasets version: 5.0.0
train rows: 157839
validation rows: 39460
test rows: 24514

First nmt_train_set item:
{'source_text': 'Tom tried to break up the fight.', 'target_text': 'Tom trató de disolver la pelea.', 'source_lang': 'eng', 'target_lang': 'spa'}


In [3]:
# Load the larger training libraries only after the datasets are ready.
import torch
import torch.nn as nn
import torch.nn.functional as F
import tokenizers
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Translator setup is ready.")

PyTorch: 2.12.0+cu126


CUDA available: True
Translator setup is ready.
